# LLM Faithfulness in Financial Sentiment Analysis
**Master's Thesis — Experiment Runner**

Each model writes its own output file (`{task}_{model}.jsonl`).  
To re-run a single failed model, just re-run its cell — it only overwrites that model's file.

**GPU requirement:**
- Llama / Gemma: T4 (16 GB) is sufficient  
- FinGPT (7B base + LoRA): A100 recommended

**Workflow:**
1. Fill in the config cell  
2. Run setup cells (mount, install, clone, login)  
3. Run experiment cells — each model in a separate cell  
4. Analyse results in `thesis_analysis.ipynb` (no GPU needed)


## Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CONFIG

REPO_URL = "https://github.com/matiimonti/llm-faithfulness-financial-sentiment-analysis.git"
REPO_DIR = "/content/thesis"
DRIVE_RESULTS = "/content/drive/MyDrive/thesis_results"

# Sample size: None = full dataset. Set e.g. 20 for a smoke test.
SAMPLE_SIZE = None

# Imports

import os
os.makedirs(DRIVE_RESULTS, exist_ok=True)

SAMPLE_ARG = f"--sample {SAMPLE_SIZE}" if SAMPLE_SIZE is not None else ""
OUTPUT_ARG = f"--output-dir {DRIVE_RESULTS}"

print(f"Results are saved to: {DRIVE_RESULTS}")

In [ ]:
!pip install -q transformers accelerate peft sentencepiece

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os, sys

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!ls {REPO_DIR}

In [ ]:
# Required for gated models (Llama-3.2, Llama-2 base for FinGPT)
from huggingface_hub import login
login()

---
## Baseline classification
One cell per model. Re-run only the cell for a model that failed.

Output files: `baseline_llama.jsonl`, `baseline_gemma.jsonl`, `baseline_fingpt.jsonl`


In [ ]:
# Baseline — Llama-3.2-3B-Instruct
!python {REPO_DIR}/experiments/run_baseline.py --models llama {SAMPLE_ARG} {OUTPUT_ARG}

In [ ]:
# Baseline — Gemma-2-2B-it
!python {REPO_DIR}/experiments/run_baseline.py --models gemma {SAMPLE_ARG} {OUTPUT_ARG}

In [ ]:
# Baseline — FinGPT-7B  (A100 recommended)
!python {REPO_DIR}/experiments/run_baseline.py --models fingpt {SAMPLE_ARG} {OUTPUT_ARG}

---
## Redaction / feature attribution (H1)

Pipeline: classify → cite key phrases → redact → re-classify.  
**Primary metric:** `faithful_destabilised_rate`

Output files: `redaction_llama.jsonl`, `redaction_gemma.jsonl`, `redaction_fingpt.jsonl`


In [ ]:
# Redaction — Llama
!python {REPO_DIR}/experiments/run_redaction.py --models llama {SAMPLE_ARG} {OUTPUT_ARG}

In [ ]:
# Redaction — Gemma
!python {REPO_DIR}/experiments/run_redaction.py --models gemma {SAMPLE_ARG} {OUTPUT_ARG}

In [ ]:
# Redaction — FinGPT  (A100 recommended)
!python {REPO_DIR}/experiments/run_redaction.py --models fingpt {SAMPLE_ARG} {OUTPUT_ARG}

---
## Counterfactual perturbation (H1)

Pipeline: classify → rewrite to opposite sentiment -> FinBERT validate -> re-classify.  
FinBERT loads once per cell and stays alive across models within that cell only.

**Primary metric:** `faithful_correct_among_valid`

Output files: `counterfactual_llama.jsonl`, `counterfactual_gemma.jsonl`, `counterfactual_fingpt.jsonl`


In [ ]:
# Counterfactual — Llama
!python {REPO_DIR}/experiments/run_counterfactual.py --models llama {SAMPLE_ARG} {OUTPUT_ARG}

In [ ]:
# Counterfactual — Gemma
!python {REPO_DIR}/experiments/run_counterfactual.py --models gemma {SAMPLE_ARG} {OUTPUT_ARG}

In [ ]:
# Counterfactual — FinGPT  (A100 recommended)
!python {REPO_DIR}/experiments/run_counterfactual.py --models fingpt {SAMPLE_ARG} {OUTPUT_ARG}

---
## CoT Intervention

Pipeline: natural CoT + prediction -> generate counter-reasoning -> original text + counter-reasoning -> new prediction.  
3 model calls per observation

**Primary metrics:** `faithful_followed_cot_rate` + `faithful_robust_rate`

Output files: `cot_intervention_llama.jsonl`, `cot_intervention_gemma.jsonl`, `cot_intervention_fingpt.jsonl`


In [ ]:
# CoT Intervention — Llama
!python {REPO_DIR}/experiments/run_cot_intervention.py --models llama {SAMPLE_ARG} {OUTPUT_ARG}

In [ ]:
# CoT Intervention — Gemma
!python {REPO_DIR}/experiments/run_cot_intervention.py --models gemma {SAMPLE_ARG} {OUTPUT_ARG}

In [ ]:
# CoT Intervention — FinGPT  (A100 recommended)
!python {REPO_DIR}/experiments/run_cot_intervention.py --models fingpt {SAMPLE_ARG} {OUTPUT_ARG}

---
## Prompt stability (H3)

3 semantically equivalent prompts per observation, greedy decoding.  
**Primary metrics:** `all_agree_rate` + BERTScore (computed in analysis notebook).

Output files: `stability_llama.jsonl`, `stability_gemma.jsonl`, `stability_fingpt.jsonl`


In [ ]:
# Stability — Llama
!python {REPO_DIR}/experiments/run_stability.py --models llama {SAMPLE_ARG} {OUTPUT_ARG}

In [ ]:
# Stability — Gemma
!python {REPO_DIR}/experiments/run_stability.py --models gemma {SAMPLE_ARG} {OUTPUT_ARG}

In [ ]:
# Stability — FinGPT  (A100 recommended)
!python {REPO_DIR}/experiments/run_stability.py --models fingpt {SAMPLE_ARG} {OUTPUT_ARG}

---
## Verify saved files


In [ ]:
import os, json

files = sorted(f for f in os.listdir(DRIVE_RESULTS) if f.endswith('.jsonl'))
print(f"Files in {DRIVE_RESULTS}:\n")
for fname in files:
    path = os.path.join(DRIVE_RESULTS, fname)
    with open(path) as fh:
        n = sum(1 for line in fh if line.strip())
    size_kb = os.path.getsize(path) / 1024
    print(f"  {fname:45s}  {n:>5} rows  {size_kb:>8.1f} KB")